# 08 — PSO Ensemble (KAGGLE GPU)
## WavSent-MTL · Tasks 3.17–3.23

**Runs on: Kaggle T4 x2 GPU**

**What this notebook does (Config G):**
- Load saved best-seed val+test predictions from notebooks 06 and 07
- Run PSO weight search on VALIDATION predictions only (not training)
- Apply learned weights to test predictions → Config G metrics
- Done separately for Kotekar and Kaggle datasets

**CRITICAL:** Individual model test metrics must be already saved (notebooks 06/07)  
**PSO uses val predictions only** — this is the key improvement over Kotekar et al.

**PREREQUISITE:** Notebooks 06 and 07 complete, all val_predictions/*.npy uploaded

In [ ]:
# ── Kaggle Setup ────────────────────────────────────────────────────────────
import subprocess
subprocess.run(['git', 'clone', 'https://github.com/IAMNEERAJ05/WavSent-MTL.git',
                '/kaggle/working/WavSent-MTL'], check=True)

import sys
import os
sys.path.insert(0, '/kaggle/working/WavSent-MTL')
os.chdir('/kaggle/working/WavSent-MTL')

import numpy as np
import pandas as pd
import json
import torch
from config.config import CONFIG

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'BEST_REPR: {CONFIG["BEST_REPR"]}')

## Copy val_predictions from Kaggle Dataset Inputs

> Upload `ablation/results/kotekar/val_predictions/` and `ablation/results/kaggle/val_predictions/`  
> as part of the Kaggle dataset, OR copy from notebook 06/07 working outputs.

In [ ]:
# If val_predictions are in the Kaggle dataset input:
KOTEKAR_PREDS_INPUT = '/kaggle/input/wavsent-kotekar-preds'
KAGGLE_PREDS_INPUT  = '/kaggle/input/wavsent-kaggle-preds'

# Copy val prediction files to expected ablation/results/ paths
import shutil

for dataset, src_dir in [('kotekar', KOTEKAR_PREDS_INPUT),
                          ('kaggle',  KAGGLE_PREDS_INPUT)]:
    dst_dir = f'ablation/results/{dataset}/val_predictions'
    os.makedirs(dst_dir, exist_ok=True)
    for model in CONFIG['pso_models']:
        for pred_type in ['val_preds', 'test_preds']:
            fname = f'{model}_{pred_type}.npy'
            src = os.path.join(src_dir, 'val_predictions', fname)
            dst = os.path.join(dst_dir, fname)
            if os.path.exists(src):
                shutil.copy2(src, dst)
                print(f'Copied: {src} -> {dst}')
            else:
                print(f'NOT FOUND: {src}  (check dataset upload)')

## PSO Helper

In [ ]:
from src.ensemble.pso_ensemble import (
    collect_val_predictions,
    run_pso_search,
    apply_ensemble_weights,
)
from src.evaluation.metrics import compute_clf_metrics


def run_ensemble_for_dataset(dataset, y_clf_val, y_clf_test):
    """Run full PSO pipeline for one dataset."""
    print(f'\n{"=" * 60}')
    print(f'PSO Ensemble: {dataset.upper()}')
    print(f'{"=" * 60}')

    # Load saved val + test predictions
    val_preds, test_preds = collect_val_predictions(dataset=dataset)

    print('Loaded val predictions:')
    for m, p in val_preds.items():
        print(f'  {m}: shape={p.shape}  mean={p.mean():.4f}')

    # Individual model test metrics (stored BEFORE ensemble)
    print('\nIndividual model test metrics (before ensemble):')
    individual_metrics = {}
    for m in CONFIG['pso_models']:
        metrics = compute_clf_metrics(y_clf_test, test_preds[m])
        individual_metrics[m] = metrics
        print(f'  {m}: acc={metrics["accuracy"]:.4f}  auc={metrics["auc"]:.4f}')

    # PSO weight search on validation predictions
    print('\nRunning PSO weight search...')
    weights = run_pso_search(
        val_preds=val_preds,
        val_labels=y_clf_val,
    )

    # Save PSO weights
    weights_path = f'/kaggle/working/pso_weights_{dataset}.json'
    with open(weights_path, 'w') as f:
        json.dump(weights, f, indent=2)
    print(f'PSO weights saved: {weights_path}')

    # Apply to test predictions → Config G
    print('\nConfig G (ensemble) test metrics:')
    g_metrics = apply_ensemble_weights(
        weights=weights,
        test_preds=test_preds,
        test_labels=y_clf_test,
    )

    # Save Config G result
    g_row = {
        'config': 'G', 'model': 'ensemble', 'seed': 0, 'run': 0,
        'dataset': dataset, **g_metrics,
        'val_accuracy': -1,  # PSO val acc stored separately
        'rmse': None, 'mae': None, 'r2': None,
    }
    result_path = f'/kaggle/working/ensemble_results_{dataset}.csv'
    pd.DataFrame([g_row]).to_csv(result_path, index=False)
    print(f'Ensemble results saved: {result_path}')

    return weights, g_metrics, individual_metrics

## Tasks 3.17–3.19 — Kotekar PSO Ensemble

In [ ]:
KOTEKAR_INPUT = '/kaggle/input/wavsent-kotekar-processed'
y_clf_val_kot  = np.load(f'{KOTEKAR_INPUT}/y_clf_val.npy')
y_clf_test_kot = np.load(f'{KOTEKAR_INPUT}/y_clf_test.npy')

kot_weights, kot_g_metrics, kot_individual = run_ensemble_for_dataset(
    dataset='kotekar',
    y_clf_val=y_clf_val_kot,
    y_clf_test=y_clf_test_kot,
)

## Tasks 3.20–3.21 — Kaggle PSO Ensemble

In [ ]:
KAGGLE_INPUT  = '/kaggle/input/wavsent-kaggle-processed'
y_clf_val_kag  = np.load(f'{KAGGLE_INPUT}/y_clf_val.npy')
y_clf_test_kag = np.load(f'{KAGGLE_INPUT}/y_clf_test.npy')

kag_weights, kag_g_metrics, kag_individual = run_ensemble_for_dataset(
    dataset='kaggle',
    y_clf_val=y_clf_val_kag,
    y_clf_test=y_clf_test_kag,
)

## Summary

In [ ]:
print('=' * 60)
print('Notebook 08 -- PSO Ensemble: COMPLETE')
print('=' * 60)

for dataset, weights, g_metrics, individual in [
    ('kotekar', kot_weights, kot_g_metrics, kot_individual),
    ('kaggle',  kag_weights, kag_g_metrics, kag_individual),
]:
    print(f'\n--- {dataset.upper()} ---')
    print(f'PSO weights: {weights}')
    print(f'Config G accuracy: {g_metrics["accuracy"]:.4f}')
    print(f'Config G AUC:      {g_metrics["auc"]:.4f}')
    print(f'Individual models:')
    for m, m_metrics in individual.items():
        print(f'  {m}: acc={m_metrics["accuracy"]:.4f}')

print()
print('Benchmark (Kotekar et al.): accuracy=0.5853, Sharpe=1.5679')
print(f'Our Config G (kotekar):    accuracy={kot_g_metrics["accuracy"]:.4f}')

print()
print('Download from /kaggle/working/:')
print('  pso_weights_kotekar.json')
print('  pso_weights_kaggle.json')
print('  ensemble_results_kotekar.csv')
print('  ensemble_results_kaggle.csv')
print()
print('Next: run notebook 09_evaluation (LOCAL)')